<a href="https://colab.research.google.com/github/JonasCandid0/TCC-Univesp/blob/main/AN%C3%81LISE_ESPACIAL_E_TEMPORAL_DE_ENCALHES_DE_FAUNA_MARINHA_NO_LITORAL_DO_ESTADO_DE_S%C3%83O_PAULO_COM_DADOS_DO_SISTEMA_SIMBA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas

In [ ]:
!pip install geopandas

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd


# Tratemento inicial
---

### Esforços de Monitoramento

In [1]:
caminho_monitoramento ='/content/drive/MyDrive/fontes de dados SIMBA/Esforcos_de_Monitoramento.csv'

In [3]:
colunas_esforco = [
    'Estado', 'Cidade', 'Praia', 'Trecho',
    'Tipo', 'Estratégia', 'Tipo de veículo',
    'Data/Hora início', 'Data/Hora término',
    'Latitude ponto inicial', 'Longitude ponto inicial',
    'Latitude ponto final', 'Longitude ponto final',
]

In [33]:
esforco_monitora_filtrada = pd.read_csv(
    caminho_monitoramento,
    sep=";",
    usecols=colunas_esforco ,
    low_memory=False
)

In [34]:
esforco_monitora_filtrada.head()

,Estado,Cidade,Praia,Trecho,Tipo,Estratégia,Tipo de veículo,Data/Hora início,Data/Hora término,Latitude ponto inicial,Longitude ponto inicial,Latitude ponto final,Longitude ponto final
0,São Paulo,Ubatuba,Caçandoquinha,Caçandoquinha,Terrestre,Diário,A pé,11/12/2025 09:05,11/12/2025 09:10,-235.650.047,-452.160.215,-235.656.297,-452.147.579
1,São Paulo,Iguape,Iguape - Praia da Juréia,Iguape - Praia da Juréia,Terrestre,Diário,Quadriciclo,11/12/2025 07:12,11/12/2025 07:29,-2.457.369,-4.724.538,-2.462.115,-4.732.833
2,São Paulo,Ubatuba,Itaipu,Itaipu,Terrestre,Diário,A pé,11/12/2025 10:13,11/12/2025 10:17,-233.789.236,-449.542.363,-233.789.307,-449.543.609
3,São Paulo,Ubatuba,Praia do Félix,Praia do Félix,Terrestre,Diário,A pé,11/12/2025 09:30,11/12/2025 10:12,-233.928.409,-44.972.134,-233.787.668,-449.547.581
4,São Paulo,Ubatuba,Vermelha do Norte,Vermelha do Norte,Terrestre,Diário,A pé,11/12/2025 08:18,11/12/2025 08:37,-234.213.601,-450.393.365,-234.133.945,-450.315.544


### Biometria

In [6]:
caminho_biometria ='/content/drive/MyDrive/fontes de dados SIMBA/Biometrias_arrumando.csv'

In [7]:
biometria = pd.read_csv(caminho_biometria, sep=";", encoding='utf-8')

In [8]:
biometria.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31440 entries, 0 to 31439
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Código                       31440 non-null  int64 
 1   Identificador da ocorrência  31440 non-null  object
 2   Identificador do indivíduo   31440 non-null  int64 
 3   Espécies - Classe            31440 non-null  object
 4   Espécies - Ordem             31438 non-null  object
 5   Espécies - Subordem          14754 non-null  object
 6   Espécies - Família           31404 non-null  object
 7   Espécies - Gênero            31322 non-null  object
 8   Espécies - Espécie           30964 non-null  object
 9   Ficha de campo               16847 non-null  object
 10  Instituição executora        31440 non-null  object
 11  Data                         31440 non-null  object
 12  Tipo                         31440 non-null  object
 13  Responsável                  31

In [31]:
colunas_selecionadas = ["Identificador da ocorrência", "Identificador do indivíduo", "Espécies - Classe",
                         "Espécies - Espécie", "Ficha de campo", "Instituição executora", "Tipo", "Responsável"
]

biometria_filtrada = biometria[colunas_selecionadas]

In [32]:
biometria_filtrada.head()

,Identificador da ocorrência,Identificador do indivíduo,Espécies - Classe,Espécies - Espécie,Ficha de campo,Instituição executora,Tipo,Responsável
0,T10RE20251211i285083,436146,Reptilia,Chelonia mydas,IA19995,Trecho 10,Quelônio,Bruno Gonçalves Pereira
1,T10MA20251211i000002,446486,Mammalia,Sotalia guianensis,IA19997,Trecho 10,Odontoceti,Guilherme Fluckiger
2,T10RE20251211i404061,432935,Reptilia,Chelonia mydas,IA19996,Trecho 10,Quelônio,André Eduardo Silva Colferai
3,T10MA20251127i404059,432933,Mammalia,Pontoporia blainvillei,IA19901,Trecho 10,Odontoceti,André Eduardo Silva Colferai
4,T10RE20251118i404058,432932,Reptilia,Chelonia mydas,IA19845,Trecho 10,Quelônio,André Eduardo Silva Colferai


### Ocorrencias de fauna alvo

In [27]:
caminho_arquivo = '/content/drive/MyDrive/fontes de dados SIMBA/O_fauna_alvo_individual.csv'

In [28]:
colunas_selecionadas = [
    'Identificador do indivíduo', 'Identificador da ocorrência',
    'Data/Hora', 'Estado', 'Cidade', 'Praia',
    'Ponto - Lat', 'Ponto - Long',
    'Espécies - Classe', 'Espécies - Espécie',
    'Estágio de desenvolvimento', 'Espécie ameaçada',
    'Condição', 'Condição da carcaça',
    'Presença de óleo', 'Interações antrópicas', 'Causa da morte'
]

In [29]:
ocorrencias_filtradas = pd.read_csv(
        caminho_arquivo,
        sep=';',
        encoding='utf-8',
        usecols=colunas_selecionadas,
        low_memory=False
    )

In [30]:
ocorrencias_filtradas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34658 entries, 0 to 34657
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Identificador do indivíduo   34658 non-null  int64 
 1   Identificador da ocorrência  34658 non-null  object
 2   Estado                       34658 non-null  object
 3   Cidade                       34658 non-null  object
 4   Praia                        34658 non-null  object
 5   Data/Hora                    34658 non-null  object
 6   Ponto - Lat                  34658 non-null  object
 7   Ponto - Long                 34658 non-null  object
 8   Espécies - Classe            34658 non-null  object
 9   Espécies - Espécie           33741 non-null  object
 10  Espécie ameaçada             33741 non-null  object
 11  Condição                     34658 non-null  object
 12  Presença de óleo             34658 non-null  object
 13  Estágio de desenvolvimento   34

In [20]:
ocorrencias_filtradas.head()

,Identificador do indivíduo,Identificador da ocorrência,Estado,Cidade,Praia,Data/Hora,Ponto - Lat,Ponto - Long,Espécies - Classe,Espécies - Espécie,Espécie ameaçada,Condição,Presença de óleo,Estágio de desenvolvimento,Condição da carcaça,Interações antrópicas,Causa da morte
0,443357,T07AV20251210i187002,São Paulo,Ilha Comprida,Ilha Comprida,10/12/2025 07:44,-249.857.182,-478.516.752,Aves,Thalasseus acuflavidus,Não,Morto,Não,Indeterminado,5,NaN,Indeterminada
1,434608,T10AV20251209i304107,São Paulo,Ilhabela,Engenho D água,09/12/2025 09:31,-237.948.417,-4.536.509,Aves,Sula leucogaster,Não,Morto,Não,Juvenil,4,NaN,Indeterminada
2,434607,T10AV20251209i304106,São Paulo,Ilhabela,Itaguassu,09/12/2025 08:55,-238.037.033,-453.649.867,Aves,Sula leucogaster,Não,Morto,Não,Juvenil,4,NaN,Indeterminada
3,443356,T07RE20251209i187191,São Paulo,Ilha Comprida,Ilha Comprida,09/12/2025 08:29,-248.492.384,-476.916.542,Reptilia,Chelonia mydas,Sim,Morto,Não,Juvenil,4,NaN,Indeterminada
4,437337,T10MA20251209i209015,São Paulo,Ubatuba,Praia do Costa,09/12/2025 07:16,-235.166.856,-451.654.592,Mammalia,Pontoporia blainvillei,Sim,Morto,Não,Filhote,3,"Tipo: Interação com pesca, Nível: 1",Indeterminada
